# 🎙️ SRT 자막 → 다국어 더빙 노트북
**공동체학습 해외 네트워크 AI Mini Seminar · 4부 실습 1** (임희석 교수, 고려대 NLP&AI 연구실)

번역 자막 파일(SRT) 하나로 **자막 시간에 맞춘 더빙 음성**과 **더빙 영상**을 만듭니다. 코딩은 필요 없습니다.

### 사용 방법
1. 위 메뉴 **[파일 → 드라이브에 사본 저장]** (내 사본에서 작업)
2. **①번 셀** 실행 (프로그램 설치)
3. **②번 셀** 실행 → [파일 선택] 버튼으로 SRT 파일(필수)과 원본 영상 MP4(선택)를 함께 올립니다.
4. **③번 셀** 실행 → 나타난 목록·슬라이더에서 목소리와 속도를 고르고 **[💾 설정 저장]** 버튼을 누릅니다. '설정 저장됨' 메시지의 목소리를 확인하세요.
5. **④번 셀** 실행 → 더빙 생성, 이어서 **⑤번**(영상 합성), **⑥번**(내려받기)을 실행합니다.

> ⚠️ [런타임 → 모두 실행]은 쓰지 마세요. ③번에서 버튼을 누르기 전에 ④번이 실행되면 멈춥니다.

결과 파일 이름은 **`원본영상이름_번역언어_trans`** 형태로 만들어집니다 (영상을 올리지 않으면 SRT 파일 이름 사용).

| 결과 파일 (예: 주일설교.mp4 → 태국어) | 내용 |
|---|---|
| `주일설교_Thai_trans.mp3` | 자막 시간에 맞춘 더빙 음성 (YouTube 오디오 트랙·HeyGen 업로드용) |
| `주일설교_Thai_trans.mp4` | 원본 영상 + 더빙 음성 (영상을 올린 경우) |
| `주일설교_Thai_trans_report.csv` | 문장별 점검표 — '⚠ 시간 초과' 문장은 대본을 줄이세요 |

> 💡 **먼저 1~2분 분량으로 시험**해 보세요. 음성은 Microsoft 신경망 음성(edge-tts)을 쓰며, 접속이 안 되면 Google 음성으로 자동 대체됩니다.
> ⚠️ AI 음성은 '초안'입니다. 배포 전 반드시 원어민이 끝까지 듣고 확인하세요.


In [ ]:
#@title ① 준비하기 (프로그램 설치 · 약 20초)
%pip install -q edge-tts numpy
%pip install -q --no-deps gTTS  # gTTS의 click 버전 고정 때문에 Colab 기본 패키지와 충돌하지 않도록 의존성 없이 설치
import shutil, sys
try:
    import google.colab  # Colab이면 성공
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if not shutil.which('ffmpeg'):
    print('⚠ ffmpeg가 없습니다. Mac: 터미널에서 brew install ffmpeg / Windows: winget install ffmpeg 후 다시 실행하세요.')
import re, os, asyncio, subprocess, csv, random, time
import numpy as np

SR = 24000  # 샘플레이트

def parse_srt(path):
    """SRT 파일을 [(번호, 시작초, 끝초, 텍스트)] 목록으로 읽기"""
    raw = open(path, 'rb').read()
    for enc in ('utf-8-sig', 'utf-16', 'cp949', 'latin-1'):
        try:
            txt = raw.decode(enc); break
        except UnicodeDecodeError:
            continue
    txt = txt.replace('\r\n', '\n').replace('\r', '\n')
    def t2s(t):
        h, m, s = t.replace('.', ',').split(':'); s, ms = s.split(',')
        return int(h) * 3600 + int(m) * 60 + int(s) + int(ms.ljust(3, '0')[:3]) / 1000
    segs = []
    for block in re.split(r'\n\s*\n', txt.strip()):
        lines = [l for l in block.split('\n') if l.strip()]
        for i, l in enumerate(lines):
            m = re.match(r'\s*(\d+:\d+:\d+[,.]\d+)\s*-->\s*(\d+:\d+:\d+[,.]\d+)', l)
            if m:
                text = ' '.join(lines[i + 1:])
                text = re.sub(r'<[^>]+>|\{[^}]+\}', '', text).strip()  # 서식 태그 제거
                if text:
                    segs.append((len(segs) + 1, t2s(m.group(1)), t2s(m.group(2)), text))
                break
    return segs

def decode(path):
    """오디오 파일 → float32 배열 (24kHz mono)"""
    out = subprocess.run(['ffmpeg', '-v', 'error', '-i', path, '-f', 'f32le', '-ac', '1', '-ar', str(SR), '-'],
                         capture_output=True, check=True).stdout
    return np.frombuffer(out, dtype=np.float32).copy()

def trim_silence(a, thr=0.01):
    idx = np.where(np.abs(a) > thr)[0]
    if len(idx) == 0: return a
    s, e = max(0, idx[0] - int(0.03 * SR)), min(len(a), idx[-1] + int(0.08 * SR))
    return a[s:e]

def speed_up(path, factor, out):
    """음높이는 유지하면서 속도만 빠르게 (ffmpeg atempo)"""
    subprocess.run(['ffmpeg', '-v', 'error', '-y', '-i', path, '-filter:a', f'atempo={factor:.3f}', out], check=True)
    return out

async def tts_edge(text, voice, rate, out):
    import edge_tts
    await edge_tts.Communicate(text, voice, rate=rate).save(out)

def tts_gtts(text, voice, out):
    from gtts import gTTS
    lang = voice.split('-')[0]
    lang = {'zh': 'zh-CN', 'fil': 'tl'}.get(lang, lang)
    gTTS(text, lang=lang).save(out)

async def synth(text, voice, rate, out, engine, tries=4):
    """Microsoft 음성으로 생성. 서버가 막으면 2·4·8초 쉬었다 재시도, 끝내 실패하면 Google 음성으로 대체"""
    if engine == 'edge':
        last = None
        for attempt in range(tries):
            try:
                await tts_edge(text, voice, rate, out); return 'edge'
            except Exception as e:
                last = e
                if attempt < tries - 1:
                    await asyncio.sleep(2 ** (attempt + 1) + random.random())
        print(f'   ⚠ Microsoft 음성 실패({last.__class__.__name__}) → Google 음성으로 대체')
    await asyncio.to_thread(tts_gtts, text, voice, out); return 'gtts'

async def make_dub(srt_path, voice, rate='+0%', max_speed=1.35, workdir='dub_work', engine='edge',
                   concurrency=7, tts_func=None, out='dub.mp3', report_path='dub_report.csv'):
    os.makedirs(workdir, exist_ok=True)
    segs = parse_srt(srt_path)
    if not segs: raise ValueError('SRT에서 자막을 찾지 못했습니다. 파일 형식을 확인하세요.')
    total = segs[-1][2] + 1.0
    print(f'자막 {len(segs)}개 · 영상 길이 약 {total/60:.1f}분 · 목소리 {voice} · 동시 생성 {concurrency}개\n')

    # 1단계: 여러 문장을 동시에 음성으로 만들기 (동시 요청 수는 concurrency로 제한)
    sem, done, used, t0 = asyncio.Semaphore(concurrency), [0], {}, time.time()
    async def one(n, text):
        f = os.path.join(workdir, f'{n:04d}.mp3')
        async with sem:
            if tts_func: await asyncio.to_thread(tts_func, text, f); used[n] = 'test'   # (테스트용)
            else: used[n] = await synth(text, voice, rate, f, engine)
        done[0] += 1
        if done[0] % 20 == 0 or done[0] == len(segs):
            print(f'   음성 생성 {done[0]}/{len(segs)}  ({time.time()-t0:.0f}초)')
    await asyncio.gather(*(one(n, text) for n, st, en, text in segs))

    # 2단계: 자막 순서대로 시간에 맞춰 배치 (결과는 한 문장씩 만든 것과 같음)
    track = np.zeros(int(total * SR) + SR, dtype=np.float32)
    report, cursor = [], 0.0
    print()
    for k, (n, st, en, text) in enumerate(segs):
        nxt = segs[k + 1][1] if k + 1 < len(segs) else en + 1.0
        slot = max(0.3, nxt - st)            # 다음 자막 시작 전까지 쓸 수 있는 시간
        f = os.path.join(workdir, f'{n:04d}.mp3')
        a = trim_silence(decode(f)); dur = len(a) / SR
        factor, note = 1.0, ''
        if dur > slot:
            factor = min(max_speed, dur / slot)
            a = trim_silence(decode(speed_up(f, factor, f.replace('.mp3', '_fast.wav'))))
            dur = len(a) / SR
            note = '속도 조절' if dur <= slot + 0.05 else '⚠ 시간 초과 — 대본을 줄이세요'
        start = max(st, cursor)              # 앞 문장과 겹치지 않게
        i0 = int(start * SR)
        if i0 + len(a) > len(track):
            track = np.concatenate([track, np.zeros(i0 + len(a) - len(track) + SR, dtype=np.float32)])
        track[i0:i0 + len(a)] += a
        cursor = start + len(a) / SR + 0.05
        delay = start - st
        if delay > 0.3 and not note: note = f'{delay:.1f}초 밀림'
        if used.get(n) == 'gtts': note = (note + ' · ' if note else '') + 'Google 음성'
        report.append([n, f'{st:.2f}', f'{slot:.2f}', f'{dur:.2f}', f'{factor:.2f}', note, text])
        if note: print(f'[{n:>3}] {st:7.2f}s  {text[:40]}{"…" if len(text) > 40 else ""}  ← {note}')
    track = np.clip(track, -1, 1)
    end = int(max(total, cursor) * SR); track = track[:end]
    subprocess.run(['ffmpeg', '-v', 'error', '-y', '-f', 'f32le', '-ar', str(SR), '-ac', '1', '-i', '-',
                    '-b:a', '160k', out], input=track.tobytes(), check=True)
    with open(report_path, 'w', newline='', encoding='utf-8-sig') as fp:
        w = csv.writer(fp); w.writerow(['번호', '시작(초)', '가능시간(초)', '음성길이(초)', '속도배율', '비고', '문장']); w.writerows(report)
    warn = sum(1 for r in report if r[5].startswith('⚠'))
    g = sum(1 for v in used.values() if v == 'gtts')
    print(f'\n✅ 완료: {out} ({len(track)/SR/60:.1f}분) · 걸린 시간 {time.time()-t0:.0f}초 · 점검표: {report_path}')
    if warn: print(f'⚠ 시간 초과 문장 {warn}개 — 점검표(csv)에서 확인 후 대본을 줄여 다시 실행하면 더 자연스럽습니다.')
    if g: print(f'⚠ {g}개 문장이 Google 음성으로 대체되어 목소리가 섞였습니다. ③번 셀에서 동시 생성 수를 줄여(예: 3) [💾 설정 저장]을 누른 뒤 ④번을 다시 실행해 보세요.')
    return out

def _video_info(path):
    """영상 코덱·픽셀 형식 확인"""
    r = subprocess.run(['ffprobe', '-v', 'error', '-select_streams', 'v:0', '-show_entries', 'stream=codec_name,pix_fmt',
                        '-of', 'csv=p=0', path], capture_output=True, text=True).stdout.strip().split(',')
    return (r[0] if r else ''), (r[1] if len(r) > 1 else '')

def merge_video(video, dub='dub.mp3', bg_volume=0.15, out='dubbed.mp4'):
    """원본 영상 + 더빙 음성 → 더빙 영상 (Mac QuickTime·iPhone·Windows 어디서나 재생되는 MP4로 저장)
    bg_volume: 원본 소리를 얼마나 남길지 (0=완전 교체)"""
    has_audio = subprocess.run(['ffprobe', '-v', 'error', '-select_streams', 'a', '-show_entries', 'stream=index',
                                '-of', 'csv=p=0', video], capture_output=True, text=True).stdout.strip() != ''
    codec, pix = _video_info(video)
    if codec == 'h264' and pix in ('yuv420p', 'yuvj420p'):
        vopt = ['-c:v', 'copy']                                   # 이미 호환되는 형식 → 그대로 복사 (빠름)
    elif codec == 'hevc' and pix in ('yuv420p', 'yuvj420p'):
        vopt = ['-c:v', 'copy', '-tag:v', 'hvc1']                 # HEVC는 Apple용 태그만 붙이면 재생됨
    else:                                                         # VP9·AV1 등 → H.264로 변환 (시간이 조금 걸림)
        print(f'ℹ️ 원본 영상 형식({codec}/{pix})을 Mac·iPhone에서도 재생되는 H.264로 변환합니다. 영상 길이에 따라 몇 분 걸릴 수 있습니다.')
        vopt = ['-c:v', 'libx264', '-preset', 'veryfast', '-crf', '20', '-pix_fmt', 'yuv420p']
    aopt = ['-c:a', 'aac', '-b:a', '160k', '-ar', '48000', '-ac', '2']
    if bg_volume > 0 and has_audio:
        fc = f'[0:a]volume={bg_volume}[bg];[1:a]volume=1.0[dub];[bg][dub]amix=inputs=2:duration=first:normalize=0[a]'
        cmd = ['ffmpeg', '-v', 'error', '-y', '-i', video, '-i', dub, '-filter_complex', fc,
               '-map', '0:v:0', '-map', '[a]'] + vopt + aopt + ['-movflags', '+faststart', out]
    else:
        cmd = ['ffmpeg', '-v', 'error', '-y', '-i', video, '-i', dub, '-map', '0:v:0', '-map', '1:a'] + vopt + aopt + ['-movflags', '+faststart', out]
    subprocess.run(cmd, check=True)
    print(f'✅ 더빙 영상 완성: {out}  (영상 {"복사" if vopt[1] == "copy" else "H.264 변환"} · 음성 AAC)')
    return out

LANG_NAMES = {'th': 'Thai', 'id': 'Indonesian', 'en': 'English', 'ko': 'Korean', 'vi': 'Vietnamese',
              'zh': 'Chinese', 'ja': 'Japanese', 'fil': 'Filipino', 'km': 'Khmer', 'my': 'Burmese',
              'lo': 'Lao', 'mn': 'Mongolian', 'hi': 'Hindi', 'es': 'Spanish'}

def output_base(video, srt, voice):
    """결과 파일 이름: '원본영상이름_번역언어_trans' (영상이 없으면 SRT 이름 사용)"""
    src = video or srt
    stem = os.path.splitext(os.path.basename(src))[0]
    stem = re.sub(r'\s*\(\d+\)$', '', stem)            # Colab이 붙이는 ' (1)' 같은 중복 번호 제거
    stem = re.sub(r'[\\/:*?"<>|]+', '_', stem).strip() or 'video'
    lang = LANG_NAMES.get(voice.split('-')[0], voice.split('-')[0])
    return f'{stem}_{lang}_trans'

class StopCell(Exception):
    """오류 메시지 없이 셀 실행만 멈추기 위한 표시"""
    def _render_traceback_(self): return []

print('✅ 준비 완료')


In [ ]:
#@title ② 파일 올리기
#@markdown SRT_PATH를 **비워 두면** 실행할 때 [파일 선택] 버튼이 나옵니다 (Colab).
#@markdown **(Colab이 아닌 VS Code·로컬에서 실행할 때만)** 파일 경로를 직접 입력하세요
SRT_PATH = "" #@param {type:"string"}
VIDEO_PATH = "" #@param {type:"string"}

if SRT_PATH:                      # VS Code·로컬: 경로 직접 입력
    SRT, VIDEO = SRT_PATH, (VIDEO_PATH or None)
else:                            # Colab: 파일 선택 버튼
    from google.colab import files
    print('👉 아래 [파일 선택] 버튼을 눌러 SRT 파일(필수)과 원본 영상 MP4(선택)를 함께 선택하세요.')
    uploaded = files.upload()
    SRT = next((f for f in uploaded if f.lower().endswith('.srt')), None)
    VIDEO = next((f for f in uploaded if f.lower().endswith(('.mp4', '.mov', '.m4v', '.mkv', '.webm'))), None)
assert SRT and os.path.exists(SRT), '❌ SRT 파일이 없습니다. (VS Code·로컬이면 위 SRT_PATH에 파일 경로를 입력하세요)'
print(f'\n자막: {SRT}\n영상: {VIDEO or "(없음 — 더빙 음성만 만듭니다)"}')
print('👉 이제 ③번 셀에서 목소리를 고르고 실행한 뒤, ④번 셀을 실행하세요.')


In [ ]:
#@title ③ 더빙 설정 — 셀을 실행한 뒤, 아래에서 값을 고르고 [💾 설정 저장] 버튼을 누르세요
import ipywidgets as W
from IPython.display import display

VOICES = ["th-TH-NiwatNeural (태국어·남)", "th-TH-PremwadeeNeural (태국어·여)", "id-ID-ArdiNeural (인도네시아어·남)", "id-ID-GadisNeural (인도네시아어·여)", "en-US-AndrewNeural (영어·남)", "en-US-AvaNeural (영어·여)", "en-AU-WilliamNeural (호주 영어·남)", "en-AU-NatashaNeural (호주 영어·여)", "ko-KR-InJoonNeural (한국어·남)", "ko-KR-SunHiNeural (한국어·여)", "vi-VN-NamMinhNeural (베트남어·남)", "vi-VN-HoaiMyNeural (베트남어·여)", "zh-CN-YunxiNeural (중국어·남)", "zh-CN-XiaoxiaoNeural (중국어·여)", "ja-JP-KeitaNeural (일본어·남)", "ja-JP-NanamiNeural (일본어·여)", "fil-PH-AngeloNeural (필리핀어·남)", "fil-PH-BlessicaNeural (필리핀어·여)", "km-KH-PisethNeural (크메르어·남)", "km-KH-SreymomNeural (크메르어·여)", "my-MM-ThihaNeural (미얀마어·남)", "my-MM-NilarNeural (미얀마어·여)", "lo-LA-ChanthavongNeural (라오어·남)", "lo-LA-KeomanyNeural (라오어·여)", "mn-MN-BataaNeural (몽골어·남)", "mn-MN-YesuiNeural (몽골어·여)", "hi-IN-MadhurNeural (힌디어·남)", "hi-IN-SwaraNeural (힌디어·여)", "es-ES-AlvaroNeural (스페인어·남)", "es-ES-ElviraNeural (스페인어·여)"]
_st = {'description_width': '170px'}
_lay = W.Layout(width='620px')
w_voice = W.Dropdown(options=VOICES, value=VOICES[0], description='더빙 목소리', style=_st, layout=_lay)
w_rate  = W.IntSlider(value=0, min=-30, max=30, step=5, description='말하기 속도 (%)', style=_st, layout=_lay)
w_max   = W.FloatSlider(value=1.35, min=1.0, max=1.6, step=0.05, readout_format='.2f', description='최대 가속 배율', style=_st, layout=_lay)
w_conc  = W.IntSlider(value=7, min=1, max=10, step=1, description='동시 생성 수', style=_st, layout=_lay)
w_bg    = W.FloatSlider(value=0.15, min=0, max=0.5, step=0.05, readout_format='.2f', description='원본 영상 소리 남기기', style=_st, layout=_lay)
btn_save = W.Button(description='💾 설정 저장', button_style='primary', layout=W.Layout(width='200px', height='40px'))
out_save = W.Output()

SETTINGS = {}   # [설정 저장]을 누를 때 여기에 저장되고, ④·⑤번 셀이 이 값을 사용합니다

def _save(_=None):
    SETTINGS.clear()
    SETTINGS.update(voice_label=w_voice.value, voice=w_voice.value.split(' ')[0],
                    rate=f"{int(w_rate.value):+d}%", max_speed=float(w_max.value),
                    concurrency=int(w_conc.value), bg_volume=float(w_bg.value), dirty=False)
    out_save.clear_output()
    with out_save:
        print('✅ 설정 저장됨')
        print(f'   목소리     : {SETTINGS["voice_label"]}')
        print(f'   말하기 속도 : {SETTINGS["rate"]} · 최대 가속: {SETTINGS["max_speed"]:.2f} · 동시 생성: {SETTINGS["concurrency"]}개')
        print(f'   원본 소리   : {SETTINGS["bg_volume"]:.2f}')
        print('👉 이제 ④번 셀을 실행하세요.')

def _changed(change):
    if SETTINGS and not SETTINGS.get('dirty'):
        SETTINGS['dirty'] = True
        out_save.clear_output()
        with out_save:
            print('✏️ 값이 바뀌었습니다. [💾 설정 저장]을 다시 눌러야 적용됩니다.')

btn_save.on_click(_save)
for _w in (w_voice, w_rate, w_max, w_conc, w_bg):
    _w.observe(_changed, names='value')

display(W.VBox([
    W.HTML('<b>① 값을 고르고 → ② [💾 설정 저장] 버튼을 누르세요.</b> '
           '<span style="color:#64748b">(최대 가속 1.35 권장 · \'Google 음성으로 대체\'가 자주 나오면 동시 생성 수를 3~5로)</span>'),
    w_voice, w_rate, w_max, w_conc, w_bg, btn_save, out_save]))


In [ ]:
#@title ④ 더빙 음성 만들기 (실행)
#@markdown ③번에서 **[💾 설정 저장]** 버튼으로 저장한 값으로 더빙을 만듭니다.
if not globals().get('SETTINGS'):
    print('❌ 아직 설정이 저장되지 않았습니다. ③번 셀을 실행하고 [💾 설정 저장] 버튼을 누른 뒤 이 셀을 다시 실행하세요.')
    raise StopCell()
if SETTINGS.get('dirty'):
    print('❌ ③번에서 값을 바꾼 뒤 저장하지 않았습니다. [💾 설정 저장]을 누른 뒤 이 셀을 다시 실행하세요.')
    raise StopCell()
OUT_BASE = output_base(VIDEO, SRT, SETTINGS['voice'])        # 예: 주일설교_Thai_trans
OUT_MP3, OUT_MP4, OUT_CSV = f'{OUT_BASE}.mp3', f'{OUT_BASE}.mp4', f'{OUT_BASE}_report.csv'
print(f'🎙️ 사용할 목소리: {SETTINGS["voice_label"]}')
print(f'📁 결과 파일 이름: {OUT_BASE}\n')
await make_dub(SRT, SETTINGS['voice'], rate=SETTINGS['rate'], max_speed=SETTINGS['max_speed'],
               concurrency=SETTINGS['concurrency'], out=OUT_MP3, report_path=OUT_CSV)
from IPython.display import Audio, display
display(Audio(OUT_MP3))


In [ ]:
#@title ⑤ 영상에 입히기 (영상을 올린 경우)
#@markdown 원본 소리 크기는 ③번 설정의 '원본 영상 소리 남기기' 값을 사용합니다.
if VIDEO:
    merge_video(VIDEO, OUT_MP3, bg_volume=SETTINGS.get('bg_volume', 0.15), out=OUT_MP4)
else:
    print(f'영상이 없어 건너뜁니다. {OUT_MP3}만 사용하세요.')


In [ ]:
#@title ⑥ 결과 내려받기
import os
out = [f for f in [OUT_MP3, OUT_MP4, OUT_CSV] if os.path.exists(f)]
try:
    import google.colab; IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    from google.colab import files
    for f in out: files.download(f)
    print('✅ 다운로드를 시작했습니다. (브라우저가 여러 파일 다운로드를 물어보면 [허용])')
else:
    for f in out: print('✅', os.path.abspath(f))


---
### 자주 묻는 질문
- **말이 너무 빠르거나 밀려요** → 점검표(`…_trans_report.csv`)에서 '⚠' 표시 문장을 AI에게 "이 문장을 N초 안에 읽을 수 있게 줄여 줘"라고 요청해 SRT를 고친 뒤 ②번부터 다시 실행.
- **발음이 이상한 이름이 있어요** → SRT에서 그 단어를 소리 나는 대로 바꿔 적으면 됩니다(예: 현지 문자로 음차).
- **목소리를 바꾸고 싶어요** → ③번 셀에서 목소리를 바꾸고 [💾 설정 저장]을 누른 뒤 ④~⑥번 실행 (파일은 다시 올리지 않아도 됩니다).
- **고른 목소리와 다른 목소리가 나와요** → ③번 셀의 '설정 저장됨' 메시지에 나온 목소리를 확인하세요. 'Google 음성으로 대체' 메시지가 있었다면 그 문장은 Google 기본 목소리(여성)로 나옵니다.
- **Mac(QuickTime)에서 '호환되지 않는 미디어' 경고가 나와요** → 최신 노트북은 원본이 VP9·AV1 등이면 자동으로 H.264로 변환해 저장합니다. 이전 버전으로 만든 파일은 ⑤번 셀을 다시 실행하거나, 그대로 VLC·Chrome으로 여세요.
- **YouTube에 음성 트랙으로 올리려면** → YouTube Studio → 언어 → 해당 언어 '더빙' 추가 → `…_trans.mp3` 업로드 (고급 기능 인증 필요).
- **HeyGen 아바타에 쓰려면** → 무료 플랜은 1분까지이므로 1분 이하 구간의 SRT만 따로 만들어 실행하세요.

*AI 음성·아바타로 만든 영상에는 설명란에 'AI 음성 더빙' 등을 표기해 주세요.*
